In [16]:
from tkinter.filedialog import askdirectory
from tkinter import *
from tkinter import messagebox
import os
import h5py

top = Tk()

top.geometry("1064x1024")


def get_override_args(file_directory, hide_defaults=True):
    vals = {}

    f1 = h5py.File(file_directory, "r")
    args = pyon.decode(f1["expid"][()])["arguments"]
    for k, v in args.items():
        if k == "ndscan_params":
            params = pyon.decode(args["ndscan_params"])
            overrides = params["overrides"]
            for k, v in overrides.items():
                spec = params["schemata"][k]["spec"]
                val = v[0]["value"]
                default = type(val)(params["schemata"][k]["default"])
                if hide_defaults and val == default:
                    continue
                vals[k] = [val / spec.get("scale", 1), spec.get("unit", "")]
        else:
            vals[k] = [v, ""]
    return vals


def args_names_and_vals(vals):
    # Print arguments names and their values
    args_names = []
    args_vals = []
    for k, (v, unit) in vals.items():
        # print(f"{k} = {v} {unit}")
        args_names.append(k)
        args_vals.append(v)
    return args_names, args_vals


# function for opening dialog for selceting the file in directory
def open_directory():
    Tk().withdraw()  # Hide the root window
    directory = askdirectory(title="Select Directory")
    if directory:
        messagebox.showinfo("Selected Directory", directory)
        if directory:
            # list files in the directory
            files = os.listdir(directory)
            file_list = "\n".join(files)
            messagebox.showinfo("Files in Directory", file_list)
    else:
        messagebox.showwarning("No Selection", "No directory selected")


# Create a button to open the directory
top.title("Open Directory and List Files")
top.geometry("300x200")


def open_file_dialog():
    directory = askdirectory(title="Select Directory")
    if directory:
        list_the_h5_files(directory)


def list_the_h5_files(directory):
    files = [f for f in os.listdir(directory) if f.endswith(".h5")]
    if files:
        file_list = "\n".join(files)
        messagebox.showinfo("H5 Files in Directory", file_list)
    else:
        messagebox.showwarning("No Selection", "No directory selected")


B = Button(top, text="Open", command=open_file_dialog)
B.place(x=50, y=50)

top.mainloop()

KeyboardInterrupt: 